# 单模型训练、单次验证与统一损失指标

金融信贷建模的常用路径：明确训练/验证/测试集，默认每个参数点训练一次；只有显式指定 `cv` 才做多折。最终模型仍为单次重训的模型。示例成本参数仅演示接口。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'hscredit').is_dir())
sys.path.insert(0, str(root))
from hscredit import LightGBM, get_loss, get_metric, list_losses, list_metrics

list_losses()

In [ ]:
list_metrics()

按放款时间分为训练、验证和独立测试集。验证集参与早停和选参，不用于最终泛化验收；测试集不参与训练。

In [ ]:
features = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '青云24']
data = pd.read_excel(root / 'examples' / 'hscredit_yyp.xlsx').sort_values('放款时间', kind='stable')
a, b = int(len(data) * .6), int(len(data) * .8)
train, valid, test = data.iloc[:a].copy(), data.iloc[a:b].copy(), data.iloc[b:].copy()
X_train, y_train = train[features], train.FPD
X_valid, y_valid = valid[features], valid.FPD
X_test, y_test = test[features], test.FPD
assert all(set(frame.FPD.unique()) == {0, 1} for frame in [train, valid, test])
pd.DataFrame([{'数据集': name, '样本数': len(frame), '坏样本率': frame.FPD.mean()} for name, frame in [('训练集', train), ('验证集', valid), ('测试集', test)]])

统一创建入口：`get_loss`、`get_metric`。同一个 `metric` 方法用 `kind` 区分代理损失和真实业务效果；指标对象可用于训练监控、搜索评分和离线验收。

In [ ]:
# 与 ModelReport.datasets 相同的容器形式，名称和顺序可检查。
valid_frame = valid[features + ['FPD']]
inputs = {
    '原生列表': [(X_valid, y_valid)],
    '具名字典': {'验证集': (X_valid, y_valid)},
    'DataFrame列表': [valid_frame],
    'DataFrame字典': {'验证集': valid_frame},
    '混合形式': {'训练监控': train[features + ['FPD']], '验证集': (X_valid, y_valid)},
    '单个二元组': (X_valid, y_valid),
    '单个DataFrame': valid_frame,
}
rows = []
reference = None
for mode, eval_data in inputs.items():
    candidate = LightGBM(target='FPD', n_estimators=3, n_jobs=1, random_state=42)
    candidate.fit(X_train, y_train, eval_set=eval_data)
    prediction = candidate.predict_proba(X_test)
    if reference is None:
        reference = prediction
    np.testing.assert_allclose(prediction, reference)
    rows.append({'输入模式': mode, '评估集名称': candidate.eval_names_})
pd.DataFrame(rows)


In [ ]:
loss = get_loss('cost_sensitive', fn_cost=5, fp_cost=1)
monitor = loss.metric()                     # 代理损失值，越小越好
metric = loss.metric(kind='business')       # 实际人均误判成本，越小越好
auc = get_metric('auc')                    # 固定坏样本概率方向
model = LightGBM(objective=loss, eval_metric=[monitor, metric, auc],
                 early_stopping_metric=monitor.name, early_stopping_rounds=5,
                 n_estimators=40, n_jobs=1, random_state=42)
# 参数已确定时直接训练；没有交叉验证。
model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])
model.evaluate(X_test, y_test, metrics=[metric, auc])

需要选参时，直接传入同一验证集。`cv=None` 为默认值，每个 point 只训练一个候选。最终模型只在训练输入上重训，验证数据不自动并入。

In [ ]:
best = model.tune(X_train, y_train, eval_set={'训练监控': (X_train, y_train), '验证集': (X_valid, y_valid)},
                  search_space={'max_depth': [2, 3], 'num_leaves': [4, 7]},
                  metric=metric, n_trials=3, n_jobs=1, retention='full')
tuner = best.tuner
assert tuner.validation_summary_['划分数'] == 1
assert best.training_summary_['样本数'] == len(train)
assert len(tuner.get_validation_predictions()) == len(valid)
tuner.validation_summary_

In [ ]:
tuner.get_optimization_history()

In [ ]:
curves = pd.concat([tuner.get_learning_curve(trial_number=trial.number) for trial in tuner.study_.trials], ignore_index=True)
curves.head(10)

In [ ]:
pd.DataFrame([{'数据集': '独立测试集', **best.evaluate(X_test, y_test, metrics=[monitor, metric, auc])}])

只有需要研究多折稳定性时才显式开启 `cv=3`。这不会把各折模型自动集成成最终模型。单次验证与多折均使用 `get_validation_predictions()` 查看验证结果；`get_fold_models()` 可显式取出保留的各折模型。

In [ ]:
cv_best = model.tune(X_train, y_train, cv=3, metric=metric,
                     search_space={'max_depth': [2], 'num_leaves': [4]},
                     n_trials=1, n_jobs=1, retention='full')
assert len(cv_best.tuner.get_fold_models()) == 3
assert (cv_best.tuner.get_validation_predictions()['验证次数'] == 1).all()
assert cv_best.training_summary_['样本数'] == len(train)
cv_best.tuner.validation_summary_

报告直接接收统一 metric 配置；下方同时核对模型原始概率、金融指标和训练过程。全部模型适配范围见 docs/articles/model-workflow.md；LR、随机森林、评分卡、规则及校准模型也使用相同评估入口。

In [ ]:
report = best.model_report(datasets={'训练集': train, '验证集': valid, '测试集': test},
                     features=features,
    metrics=[monitor, metric, auc], target='FPD', verbose=False, n_jobs=1)
np.testing.assert_allclose(report._datasets['测试集'].prediction, best.predict_proba(X_test)[:, 1])
report.get_metrics()
report.get_training_process()